# Malaika — Run the Full App on Colab

This notebook launches the complete Malaika IMCI assessment app with Gemma 4 E4B.

**Requirements:** Colab with T4 GPU (free tier works).

**What happens:**
1. Clones the repo
2. Installs dependencies
3. Downloads the merged fine-tuned model from HuggingFace
4. Launches Gradio app with public URL
5. Open the URL on your phone — full IMCI assessment, offline-capable

In [ ]:
# Cell 1: Clone repo + install deps
!git clone https://github.com/klickgenai/deepmind-hackathon.git /content/malaika
%cd /content/malaika
!pip install -q -r requirements.txt
!pip install -q peft  # For LoRA adapter loading

In [ ]:
# Cell 2: Authenticate with HuggingFace (for private merged model)
from huggingface_hub import login
import os

# Option A: Use Colab secrets
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None

if hf_token:
    login(token=hf_token)
    print("Authenticated via Colab secrets")
else:
    # Option B: Manual login
    login()
    print("Authenticated manually")

In [ ]:
# Cell 3: GPU check
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Cell 4: Configure app to use merged model from HuggingFace
# Override the default model to use our fine-tuned merged version
import json
from pathlib import Path

# The merged model has LoRA baked in — no separate adapter needed
MERGED_MODEL = "Vimal0703/malaika-breath-sounds-E4B-merged"
BASE_MODEL = "google/gemma-4-E4B-it"  # Fallback if merged unavailable

# Check if merged model is accessible
from huggingface_hub import repo_exists
if repo_exists(MERGED_MODEL):
    USE_MODEL = MERGED_MODEL
    print(f"Using merged fine-tuned model: {MERGED_MODEL}")
else:
    USE_MODEL = BASE_MODEL
    print(f"Merged model not found, using base: {BASE_MODEL}")

print(f"\nModel will be downloaded on first load (~5-10 GB)")

In [ ]:
# Cell 5: Patch config to use the right model, then launch app
import sys
sys.path.insert(0, '/content/malaika')

from malaika.config import MalaikaConfig, ModelConfig, load_config

# Override config with our model
config = load_config()
config.model.model_name = USE_MODEL

# If using merged model, disable separate LoRA loading (already baked in)
if USE_MODEL == MERGED_MODEL:
    config.model.enable_breath_sounds_adapter = False

print(f"Config:")
print(f"  Model: {config.model.model_name}")
print(f"  4-bit: {config.model.quantize_4bit}")
print(f"  LoRA adapter: {'disabled (merged)' if not config.model.enable_breath_sounds_adapter else 'enabled'}")
print(f"  TTS: {config.features.enable_tts}")
print(f"  Heart module: {config.features.enable_heart_rate}")

In [ ]:
# Cell 6: Load the model (downloads ~5-10 GB on first run)
from malaika.inference import MalaikaInference

inference = MalaikaInference(config)
print("Loading model... (this takes 2-5 minutes on first run)")
inference.load_model()
print(f"Model loaded on {inference.device}!")
print(f"VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 7: Quick smoke test — verify model generates reasonable output
test_messages = [
    {"role": "user", "content": "You are a medical AI. Say 'Malaika ready' if you can help with child health assessment."}
]
response = inference.generate(test_messages, max_tokens=50, temperature=0.0)
print(f"Smoke test response: {response[:200]}")
print("Model is working!" if len(response) > 0 else "WARNING: Empty response")

In [ ]:
# Cell 8: Launch the Gradio app with public URL
from malaika.app import create_app

# Monkey-patch to use our already-loaded model instead of re-loading
import malaika.app as app_module

# Store inference in app state
_original_init = app_module.AppState.__init__
def _patched_init(self, cfg):
    _original_init(self, cfg)
    self.inference = inference
    self.model_loaded = True
    self.model_error = None
app_module.AppState.__init__ = _patched_init

app = create_app(config)
print("\nLaunching Malaika...")
print("Open the public URL below on your phone to test!")
app.launch(
    share=True,
    server_name="0.0.0.0",
    server_port=7860,
    show_error=True,
)